# Donut EDS Adapter - Production Ready

This notebook processes Executive Document Summary (EDS) forms from PDF documents using Donut.

## Key Features:
1. **Hierarchical Query Structure**: 8 organized extraction passes
2. **Testing Mode**: Quick validation on example forms
3. **Progress Tracking**: Visual progress bar with ETA
4. **Logging**: All output saved to timestamped log file
5. **Resume Capability**: Automatically skip processed files
6. **Memory Management**: GPU cache clearing to prevent OOM
7. **Output Validation**: JSON integrity checks
8. **Quiet Operation**: Minimal console spam

## Quick Start:
- **Testing:** Set `TESTING_MODE = True`, add PDFs to `_exampleforms/`
- **Production:** Set `TESTING_MODE = False`

In [ ]:
import pandas as pd
import os
from pathlib import Path
import json
from typing import List, Dict, Any, Optional, Tuple
import PyPDF2
from io import BytesIO
import base64
from PIL import Image
import fitz  # PyMuPDF
import torch
from transformers import DonutProcessor, VisionEncoderDecoderModel
import re
from datetime import datetime
from decimal import Decimal, InvalidOperation
import glob
import logging
from tqdm import tqdm
import warnings

# Suppress transformer warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message='.*valid.*ignored.*')

## Configuration

In [ ]:
# ============================================
# MAIN CONFIGURATION
# ============================================

# TESTING MODE - Set to True to use example forms for quick testing
TESTING_MODE = True  # <-- Set to False for production runs

# VERBOSITY - Set to True to see detailed extraction logs
VERBOSE = False  # <-- Set to True for debugging, False for quiet operation

# Processing options
CLOBBER = False  # Set to True to overwrite existing results
FILTER_AGENCIES = None  # Set to agency name(s) or None for all (only for production mode)

# Testing mode sample size (None = process all files in _exampleforms/)
TESTING_SAMPLE_SIZE = None  # Set to 3, 5, 10 etc. for quick iteration

# File paths - PRODUCTION MODE
CSV_PATH = "../../code/preprocessing/zero_shot_results_full_corpus.csv"
CONTRACTS_DIR = "../../data/raw/_contracts/"
OUTPUT_DIR = "../../data/intermediate_products/eds_forms_donut_improved/"

# File paths - TESTING MODE
TESTING_DIR = "../../data/raw/_exampleforms/"
TESTING_OUTPUT_DIR = "../../data/intermediate_products/eds_forms_donut_testing/"

# Donut configuration
MODEL_NAME = "naver-clova-ix/donut-base-finetuned-docvqa"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LENGTH = 512
IMAGE_SIZE = [1280, 960]
DPI = 300  # Increased from 200 for better image quality

# ============================================
# LOGGING SETUP
# ============================================

# Create logs directory
logs_dir = Path("../../logs")
logs_dir.mkdir(parents=True, exist_ok=True)

# Setup logging with timestamp
log_filename = logs_dir / f"donut_processing_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_filename),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger(__name__)

# ============================================
# AUTO-CONFIGURATION BASED ON MODE
# ============================================

if TESTING_MODE:
    logger.info("=" * 50)
    logger.info("🧪 TESTING MODE ENABLED")
    logger.info("=" * 50)
    logger.info(f"Source: {TESTING_DIR}")
    logger.info(f"Output: {TESTING_OUTPUT_DIR}")
    logger.info(f"DPI: {DPI}")
    if TESTING_SAMPLE_SIZE:
        logger.info(f"Sample Size: {TESTING_SAMPLE_SIZE} files")
    logger.info("=" * 50)
    ACTIVE_OUTPUT_DIR = TESTING_OUTPUT_DIR
else:
    logger.info("=" * 50)
    logger.info("🚀 PRODUCTION MODE")
    logger.info("=" * 50)
    logger.info(f"Source: {CONTRACTS_DIR}")
    logger.info(f"Output: {OUTPUT_DIR}")
    logger.info(f"DPI: {DPI}")
    if FILTER_AGENCIES:
        logger.info(f"Agency Filter: {FILTER_AGENCIES}")
    logger.info("=" * 50)
    ACTIVE_OUTPUT_DIR = OUTPUT_DIR

logger.info(f"Log file: {log_filename}")
logger.info(f"Verbose mode: {VERBOSE}")
logger.info(f"Using device: {DEVICE}")

if DEVICE == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    logger.info(f"GPU: {gpu_name}")
    logger.info(f"GPU Memory: {gpu_memory:.1f} GB")

## Define Structured Schema and Query Strategy

In [ ]:
# Define the expected output schema
OUTPUT_SCHEMA = {
    "eds_number": "string",
    "date_prepared": "date",
    "agency": {
        "name": "string",
        "address": "string",
        "contact_person": "string",
        "telephone": "string",
        "email": "string"
    },
    "contract_info": {
        "professional_personal_services": "boolean",
        "grant": "boolean",
        "lease": "boolean",
        "attorney": "boolean",
        "mou": "boolean",
        "qpa": "boolean",
        "qpa_details": "string",
        "contract_for_procured_services": "boolean",
        "maintenance": "boolean",
        "license_agreement": "boolean",
        "amendment_number": "string",
        "renewal_number": "string",
        "other": "boolean",
        "other_details": "string"
    },
    "vendor": {
        "id": "string",
        "name": "string",
        "address": "string",
        "telephone": "string",
        "email": "string",
        "registered_with_sos": "boolean",
        "minority_owned": "boolean",
        "minority_percentage": "string",
        "women_owned": "boolean",
        "women_percentage": "string"
    },
    "fiscal": {
        "account_number": "string",
        "account_name": "string",
        "amount_this_action": "float",
        "new_contract_total": "float",
        "revenue_generated_this_action": "float",
        "revenue_generated_total": "float",
        "amounts_by_year": []
    },
    "time_period": {
        "from_date": "date",
        "to_date": "date"
    },
    "source_selection": {
        "bid_quotation": "boolean",
        "rfp_number": "string",
        "emergency": "boolean",
        "negotiated": "boolean",
        "special_procurement": "boolean",
        "other": "boolean",
        "other_details": "string"
    },
    "additional_info": {
        "renewal_language": "boolean",
        "termination_for_convenience": "boolean",
        "description_of_work": "string",
        "vendor_justification": "string"
    }
}

# Hierarchical query structure - organized by extraction pass
QUERY_STRUCTURE = {
    "pass1_basic_identification": [
        "What is the EDS Number?",
        "What is the Date Prepared?",
        "What is the Name of agency from section 14?"
    ],
    
    "pass2_agency_contact": [
        "What is the contact person name from the AGENCY CONTACT INFORMATION section?",
        "What is the telephone number from the AGENCY CONTACT INFORMATION section?",
        "What is the E-mail address from the AGENCY CONTACT INFORMATION section?"
    ],
    
    "pass3_contract_types": [
        "In section 3 CONTRACTS & LEASES, is Professional/Personal Services checked?",
        "In section 3, is Grant checked?",
        "In section 3, is Lease checked?",
        "In section 3, is Attorney checked?",
        "In section 3, is MOU checked?",
        "In section 3, is QPA checked?",
        "In section 3, is Contract for procured Services checked?",
        "In section 3, is Maintenance checked?",
        "In section 3, is License Agreement checked?",
        "In section 3, what is the Amendment number?",
        "In section 3, what is the Renewal number?",
        "In section 3, is Other checked? If yes, what is written?"
    ],
    
    "pass4_vendor_info": [
        "What is the Vendor ID number from section 23?",
        "What is the Vendor Name from section 24?",
        "What is the vendor telephone number from section 25?",
        "What is the vendor E-mail address from section 27?",
        "What is the vendor address from section 26?",
        "Is Yes checked for section 28 - Is the vendor registered with the Secretary of State?",
        "Is Yes checked for Primary Vendor Minority in section 29?",
        "What is the percentage for Primary Vendor Minority in section 30?",
        "Is Yes checked for Primary Vendor Women in section 29?",
        "What is the percentage for Primary Vendor Women in section 30?"
    ],
    
    "pass5_fiscal_info": [
        "What is the Account Number from section 4?",
        "What is the Account Name from section 5?",
        "What is the Total amount this action from section 6?",
        "What is the New contract total from section 7?",
        "What is the Revenue generated this action from section 8?",
        "What is the Revenue generated total contract from section 9?"
    ],
    
    "pass6_dates": [
        "What is the From date from section 11?",
        "What is the To date from section 12?"
    ],
    
    "pass7_source_selection": [
        "In section 13 Method of source selection, is Bid/Quotation selected?",
        "In section 13, what is the RFP number if any?",
        "In section 13, is Emergency selected?",
        "In section 13, is Negotiated selected?",
        "In section 13, is Special Procurement selected?",
        "In section 13, is Other selected? If yes, what is specified?"
    ],
    
    "pass8_additional_fields": [
        "In section 33, is there Renewal Language in the document?",
        "In section 34, is there a Termination for Convenience clause?"
    ]
}

total_queries = sum(len(queries) for queries in QUERY_STRUCTURE.values())
logger.info(f"Query structure: {len(QUERY_STRUCTURE)} passes, {total_queries} total queries")

## Load Donut Model

In [ ]:
logger.info("Loading Donut processor and model...")
processor = DonutProcessor.from_pretrained(MODEL_NAME, use_fast=True)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)

if DEVICE == "cuda":
    model.half()  # Use half precision for faster inference
    
model.to(DEVICE)
model.eval()

logger.info(f"Model loaded successfully on {DEVICE}")

## Load and Filter Data

In [ ]:
if TESTING_MODE:
    logger.info("\n=== TESTING MODE: Loading Example Forms ===")
    
    # Find all PDF files in the testing directory
    testing_path = Path(TESTING_DIR)
    pdf_files = list(testing_path.glob("*.pdf"))
    
    if not pdf_files:
        logger.warning(f"⚠️  No PDF files found in {TESTING_DIR}")
        logger.warning("Please add example PDF files to the testing directory.")
    else:
        logger.info(f"Found {len(pdf_files)} PDF files")
    
    # Create a simple dataframe for testing
    forms_df = pd.DataFrame([
        {
            'filename': pdf_file.name,
            'full_path': str(pdf_file),
            'form_pages': '1',  # Assume first page for testing
            'num_form_pages': 1,
            'lowest_page': 1
        }
        for pdf_file in pdf_files
    ])
    
    # Apply sample size limit if specified
    if TESTING_SAMPLE_SIZE and len(forms_df) > TESTING_SAMPLE_SIZE:
        logger.info(f"Limiting to first {TESTING_SAMPLE_SIZE} files for quick testing")
        forms_df = forms_df.head(TESTING_SAMPLE_SIZE)
    
    logger.info(f"Prepared {len(forms_df)} documents for testing")
    
else:
    logger.info("\n=== PRODUCTION MODE: Loading from CSV ===")
    
    # Load professional services contracts data
    with open("../../data/raw/indiana_prof_services_contracts.json", 'r') as f:
        prof_services_contracts = json.load(f)
    
    logger.info(f"Total professional services contracts: {len(prof_services_contracts)}")
    
    # Apply agency filter if specified
    if FILTER_AGENCIES is not None:
        logger.info(f"=== AGENCY FILTER APPLIED ===")
        target_agencies = [FILTER_AGENCIES] if isinstance(FILTER_AGENCIES, str) else FILTER_AGENCIES
        logger.info(f"Filtering for agencies: {target_agencies}")
        
        original_count = len(prof_services_contracts)
        prof_services_contracts = [
            contract for contract in prof_services_contracts 
            if contract['agencyName'] in target_agencies
        ]
        logger.info(f"Contracts after filter: {len(prof_services_contracts)} (from {original_count})")
    
    # Extract filenames
    prof_services_filenames = set()
    for contract in prof_services_contracts:
        filename = contract['pdfUrl'].split('/')[-1]
        prof_services_filenames.add(filename)
    
    logger.info(f"Unique professional services PDF files: {len(prof_services_filenames)}")
    
    # Load CSV and filter for professional services with forms
    df = pd.read_csv(CSV_PATH, low_memory=False)
    logger.info(f"Total documents in CSV: {len(df)}")
    
    forms_df = df[df['contains_form'] == True].copy()
    logger.info(f"Documents with forms: {len(forms_df)}")
    
    forms_df = forms_df[forms_df['filename'].isin(prof_services_filenames)].copy()
    logger.info(f"Professional services documents with forms: {len(forms_df)}")

## Helper Functions - Post-Processing and Validation

In [ ]:
def normalize_boolean_response(answer: str) -> Optional[bool]:
    """
    Normalize various boolean responses to True/False/None.
    Handles: yes, no, x, checked, unchecked, etc.
    """
    if not answer or answer.strip() == "":
        return None
    
    answer_lower = answer.strip().lower()
    
    # Positive indicators
    if any(x in answer_lower for x in ['yes', 'checked', 'true', 'x']):
        # Make sure it's not "no" or "unchecked"
        if 'no' not in answer_lower and 'un' not in answer_lower:
            return True
    
    # Negative indicators
    if any(x in answer_lower for x in ['no', 'unchecked', 'false']):
        return False
    
    return None


def parse_currency(amount_str: str) -> Optional[float]:
    """
    Parse currency strings to float.
    Handles: $1,234.56, 1234.56, $1234, etc.
    """
    if not amount_str or amount_str.strip() == "":
        return None
    
    try:
        # Remove currency symbols, commas, and spaces
        cleaned = re.sub(r'[$,\s]', '', amount_str.strip())
        return float(cleaned)
    except (ValueError, InvalidOperation):
        return None


def normalize_date(date_str: str) -> Optional[str]:
    """
    Normalize date strings to ISO format (YYYY-MM-DD).
    Handles various formats: MM/DD/YYYY, M/D/YY, Month DD, YYYY, etc.
    """
    if not date_str or date_str.strip() == "":
        return None
    
    date_str = date_str.strip()
    
    # Common patterns
    patterns = [
        r'(\d{1,2})/(\d{1,2})/(\d{4})',  # MM/DD/YYYY
        r'(\d{1,2})/(\d{1,2})/(\d{2})',   # MM/DD/YY
        r'(\d{4})-(\d{2})-(\d{2})',       # YYYY-MM-DD
    ]
    
    for pattern in patterns:
        match = re.search(pattern, date_str)
        if match:
            try:
                if pattern == patterns[0]:  # MM/DD/YYYY
                    month, day, year = match.groups()
                    return f"{year}-{month.zfill(2)}-{day.zfill(2)}"
                elif pattern == patterns[1]:  # MM/DD/YY
                    month, day, year = match.groups()
                    year = f"20{year}" if int(year) < 50 else f"19{year}"
                    return f"{year}-{month.zfill(2)}-{day.zfill(2)}"
                elif pattern == patterns[2]:  # Already ISO
                    return match.group(0)
            except ValueError:
                continue
    
    # Try natural language parsing for month names
    try:
        from dateutil import parser
        parsed = parser.parse(date_str, fuzzy=True)
        return parsed.strftime('%Y-%m-%d')
    except:
        pass
    
    return None


def extract_amendment_renewal_number(answer: str) -> Optional[str]:
    """
    Extract amendment or renewal number from answer.
    """
    if not answer or answer.strip() == "":
        return None
    
    # Look for number patterns
    number_match = re.search(r'\b(\d+)\b', answer)
    if number_match:
        return number_match.group(1)
    
    return None


def validate_fiscal_consistency(fiscal_data: Dict) -> Dict:
    """
    Validate fiscal data consistency and add warnings if inconsistent.
    """
    warnings = []
    
    # Check if year amounts sum to contract total
    if fiscal_data.get('amounts_by_year'):
        year_sum = sum(year['amount'] for year in fiscal_data['amounts_by_year'] if year['amount'])
        contract_total = fiscal_data.get('new_contract_total')
        
        if contract_total and year_sum > 0:
            difference = abs(year_sum - contract_total)
            if difference > 1.0:  # Allow for rounding errors
                warnings.append(f"Year amounts ({year_sum}) don't match contract total ({contract_total})")
    
    fiscal_data['validation_warnings'] = warnings
    return fiscal_data


def validate_json_output(json_path: Path) -> bool:
    """
    Validate that saved JSON is well-formed and contains required fields.
    Returns True if valid, False otherwise.
    """
    try:
        with open(json_path, 'r') as f:
            data = json.load(f)
        
        # Check for error field
        if 'error' in data:
            return True  # Error files are valid, just note the error
        
        # Check for required fields
        if 'structured_data' not in data:
            logger.warning(f"Missing structured_data in {json_path.name}")
            return False
        
        return True
        
    except json.JSONDecodeError as e:
        logger.error(f"Invalid JSON in {json_path.name}: {e}")
        return False
    except Exception as e:
        logger.error(f"Error validating {json_path.name}: {e}")
        return False


logger.info("Post-processing and validation functions loaded")

## Core Processing Functions

In [ ]:
def get_lowest_page_number(form_pages_str: str) -> Optional[int]:
    """Extract the lowest page number from form_pages string."""
    if pd.isna(form_pages_str) or form_pages_str == "":
        return None
    
    if ',' in str(form_pages_str):
        page_numbers = [int(x.strip()) for x in str(form_pages_str).split(',')]
    else:
        page_numbers = [int(str(form_pages_str).strip())]
    
    return min(page_numbers)


def pdf_page_to_image(pdf_path: str, page_number: int, dpi: int = DPI) -> Image.Image:
    """Convert a single PDF page to PIL Image."""
    doc = fitz.open(pdf_path)
    page = doc.load_page(page_number - 1)
    
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    pix = page.get_pixmap(matrix=mat)
    img_data = pix.tobytes("ppm")
    
    doc.close()
    
    return Image.open(BytesIO(img_data))


def query_donut(image: Image.Image, query: str) -> str:
    """
    Query Donut model with a single question.
    """
    task_prompt = f"<s_docvqa><s_question>{query}</s_question><s_answer>"
    
    pixel_values = processor(image, return_tensors="pt").pixel_values
    
    if DEVICE == "cuda":
        pixel_values = pixel_values.half()
    
    pixel_values = pixel_values.to(DEVICE)
    
    decoder_input_ids = processor.tokenizer(
        task_prompt,
        add_special_tokens=False,
        return_tensors="pt"
    ).input_ids.to(DEVICE)
    
    with torch.no_grad():
        outputs = model.generate(
            pixel_values,
            decoder_input_ids=decoder_input_ids,
            max_length=MAX_LENGTH,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
            use_cache=True,
            num_beams=1,
            bad_words_ids=[[processor.tokenizer.unk_token_id]],
            return_dict_in_generate=True
        )
    
    sequence = processor.batch_decode(outputs.sequences)[0]
    answer = sequence.replace(processor.tokenizer.eos_token, "").replace(processor.tokenizer.pad_token, "")
    
    # Parse answer
    answer_match = re.search(r'<s_answer>(.*?)</s_answer>', answer)
    if answer_match:
        return answer_match.group(1).strip()
    else:
        answer_start = answer.find('<s_answer>') + len('<s_answer>')
        return answer[answer_start:].strip()


logger.info("Core processing functions loaded")

## Multi-Pass Hierarchical Extraction

In [ ]:
def process_with_donut_hierarchical(image: Image.Image, filename: str = "") -> Dict[str, Any]:
    """
    Process document with hierarchical multi-pass extraction.
    Quiet operation unless VERBOSE=True.
    """
    results = {
        'processing_timestamp': datetime.now().isoformat(),
        'model_name': MODEL_NAME,
        'device': DEVICE,
        'dpi': DPI,
        'testing_mode': TESTING_MODE,
        'extraction_passes': {},
        'structured_data': {},
        'raw_qa_pairs': []
    }
    
    try:
        # Pass 1: Basic Identification
        if VERBOSE:
            logger.info(f"  [{filename}] Pass 1/8: Basic Identification")
        pass1_results = {}
        for query in QUERY_STRUCTURE['pass1_basic_identification']:
            answer = query_donut(image, query)
            pass1_results[query] = answer
            results['raw_qa_pairs'].append({'query': query, 'answer': answer})
        
        results['extraction_passes']['pass1_basic_identification'] = pass1_results
        results['structured_data']['eds_number'] = pass1_results.get("What is the EDS Number?", "")
        results['structured_data']['date_prepared'] = normalize_date(pass1_results.get("What is the Date Prepared?", ""))
        
        # Pass 2: Agency Contact
        if VERBOSE:
            logger.info(f"  [{filename}] Pass 2/8: Agency Contact")
        pass2_results = {}
        for query in QUERY_STRUCTURE['pass2_agency_contact']:
            answer = query_donut(image, query)
            pass2_results[query] = answer
            results['raw_qa_pairs'].append({'query': query, 'answer': answer})
        
        results['extraction_passes']['pass2_agency_contact'] = pass2_results
        results['structured_data']['agency'] = {
            'name': pass1_results.get("What is the Name of agency from section 14?", ""),
            'contact_person': pass2_results.get("What is the contact person name from the AGENCY CONTACT INFORMATION section?", ""),
            'telephone': pass2_results.get("What is the telephone number from the AGENCY CONTACT INFORMATION section?", ""),
            'email': pass2_results.get("What is the E-mail address from the AGENCY CONTACT INFORMATION section?", "")
        }
        
        # Pass 3: Contract Types
        if VERBOSE:
            logger.info(f"  [{filename}] Pass 3/8: Contract Types")
        pass3_results = {}
        for query in QUERY_STRUCTURE['pass3_contract_types']:
            answer = query_donut(image, query)
            pass3_results[query] = answer
            results['raw_qa_pairs'].append({'query': query, 'answer': answer})
        
        results['extraction_passes']['pass3_contract_types'] = pass3_results
        results['structured_data']['contract_info'] = {
            'professional_personal_services': normalize_boolean_response(pass3_results.get("In section 3 CONTRACTS & LEASES, is Professional/Personal Services checked?", "")),
            'grant': normalize_boolean_response(pass3_results.get("In section 3, is Grant checked?", "")),
            'lease': normalize_boolean_response(pass3_results.get("In section 3, is Lease checked?", "")),
            'attorney': normalize_boolean_response(pass3_results.get("In section 3, is Attorney checked?", "")),
            'mou': normalize_boolean_response(pass3_results.get("In section 3, is MOU checked?", "")),
            'qpa': normalize_boolean_response(pass3_results.get("In section 3, is QPA checked?", "")),
            'contract_for_procured_services': normalize_boolean_response(pass3_results.get("In section 3, is Contract for procured Services checked?", "")),
            'maintenance': normalize_boolean_response(pass3_results.get("In section 3, is Maintenance checked?", "")),
            'license_agreement': normalize_boolean_response(pass3_results.get("In section 3, is License Agreement checked?", "")),
            'amendment_number': extract_amendment_renewal_number(pass3_results.get("In section 3, what is the Amendment number?", "")),
            'renewal_number': extract_amendment_renewal_number(pass3_results.get("In section 3, what is the Renewal number?", "")),
            'other': normalize_boolean_response(pass3_results.get("In section 3, is Other checked? If yes, what is written?", "")),
            'other_details': pass3_results.get("In section 3, is Other checked? If yes, what is written?", "")
        }
        
        # Pass 4: Vendor Information
        if VERBOSE:
            logger.info(f"  [{filename}] Pass 4/8: Vendor Information")
        pass4_results = {}
        for query in QUERY_STRUCTURE['pass4_vendor_info']:
            answer = query_donut(image, query)
            pass4_results[query] = answer
            results['raw_qa_pairs'].append({'query': query, 'answer': answer})
        
        results['extraction_passes']['pass4_vendor_info'] = pass4_results
        results['structured_data']['vendor'] = {
            'id': pass4_results.get("What is the Vendor ID number from section 23?", ""),
            'name': pass4_results.get("What is the Vendor Name from section 24?", ""),
            'telephone': pass4_results.get("What is the vendor telephone number from section 25?", ""),
            'email': pass4_results.get("What is the vendor E-mail address from section 27?", ""),
            'address': pass4_results.get("What is the vendor address from section 26?", ""),
            'registered_with_sos': normalize_boolean_response(pass4_results.get("Is Yes checked for section 28 - Is the vendor registered with the Secretary of State?", "")),
            'minority_owned': normalize_boolean_response(pass4_results.get("Is Yes checked for Primary Vendor Minority in section 29?", "")),
            'minority_percentage': pass4_results.get("What is the percentage for Primary Vendor Minority in section 30?", ""),
            'women_owned': normalize_boolean_response(pass4_results.get("Is Yes checked for Primary Vendor Women in section 29?", "")),
            'women_percentage': pass4_results.get("What is the percentage for Primary Vendor Women in section 30?", "")
        }
        
        # Pass 5: Fiscal Information
        if VERBOSE:
            logger.info(f"  [{filename}] Pass 5/8: Fiscal Information")
        pass5_results = {}
        for query in QUERY_STRUCTURE['pass5_fiscal_info']:
            answer = query_donut(image, query)
            pass5_results[query] = answer
            results['raw_qa_pairs'].append({'query': query, 'answer': answer})
        
        results['extraction_passes']['pass5_fiscal_info'] = pass5_results
        fiscal_data = {
            'account_number': pass5_results.get("What is the Account Number from section 4?", ""),
            'account_name': pass5_results.get("What is the Account Name from section 5?", ""),
            'amount_this_action': parse_currency(pass5_results.get("What is the Total amount this action from section 6?", "")),
            'new_contract_total': parse_currency(pass5_results.get("What is the New contract total from section 7?", "")),
            'revenue_generated_this_action': parse_currency(pass5_results.get("What is the Revenue generated this action from section 8?", "")),
            'revenue_generated_total': parse_currency(pass5_results.get("What is the Revenue generated total contract from section 9?", "")),
            'amounts_by_year': []
        }
        results['structured_data']['fiscal'] = validate_fiscal_consistency(fiscal_data)
        
        # Pass 6: Dates
        if VERBOSE:
            logger.info(f"  [{filename}] Pass 6/8: Contract Dates")
        pass6_results = {}
        for query in QUERY_STRUCTURE['pass6_dates']:
            answer = query_donut(image, query)
            pass6_results[query] = answer
            results['raw_qa_pairs'].append({'query': query, 'answer': answer})
        
        results['extraction_passes']['pass6_dates'] = pass6_results
        results['structured_data']['time_period'] = {
            'from_date': normalize_date(pass6_results.get("What is the From date from section 11?", "")),
            'to_date': normalize_date(pass6_results.get("What is the To date from section 12?", ""))
        }
        
        # Pass 7: Source Selection
        if VERBOSE:
            logger.info(f"  [{filename}] Pass 7/8: Source Selection Method")
        pass7_results = {}
        for query in QUERY_STRUCTURE['pass7_source_selection']:
            answer = query_donut(image, query)
            pass7_results[query] = answer
            results['raw_qa_pairs'].append({'query': query, 'answer': answer})
        
        results['extraction_passes']['pass7_source_selection'] = pass7_results
        results['structured_data']['source_selection'] = {
            'bid_quotation': normalize_boolean_response(pass7_results.get("In section 13 Method of source selection, is Bid/Quotation selected?", "")),
            'rfp_number': pass7_results.get("In section 13, what is the RFP number if any?", ""),
            'emergency': normalize_boolean_response(pass7_results.get("In section 13, is Emergency selected?", "")),
            'negotiated': normalize_boolean_response(pass7_results.get("In section 13, is Negotiated selected?", "")),
            'special_procurement': normalize_boolean_response(pass7_results.get("In section 13, is Special Procurement selected?", "")),
            'other': normalize_boolean_response(pass7_results.get("In section 13, is Other selected? If yes, what is specified?", "")),
            'other_details': pass7_results.get("In section 13, is Other selected? If yes, what is specified?", "")
        }
        
        # Pass 8: Additional Fields
        if VERBOSE:
            logger.info(f"  [{filename}] Pass 8/8: Additional Fields")
        pass8_results = {}
        for query in QUERY_STRUCTURE['pass8_additional_fields']:
            answer = query_donut(image, query)
            pass8_results[query] = answer
            results['raw_qa_pairs'].append({'query': query, 'answer': answer})
        
        results['extraction_passes']['pass8_additional_fields'] = pass8_results
        results['structured_data']['additional_info'] = {
            'renewal_language': normalize_boolean_response(pass8_results.get("In section 33, is there Renewal Language in the document?", "")),
            'termination_for_convenience': normalize_boolean_response(pass8_results.get("In section 34, is there a Termination for Convenience clause?", ""))
        }
        
        if VERBOSE:
            logger.info(f"  [{filename}] ✓ Extraction complete")
        
    except Exception as e:
        logger.error(f"  [{filename}] ✗ Error during extraction: {str(e)}")
        results['error'] = str(e)
    
    return results


logger.info("Hierarchical extraction function loaded")

## Prepare Processing Queue

In [ ]:
if not TESTING_MODE:
    # Production mode: Add lowest page number
    forms_df['lowest_page'] = forms_df['form_pages'].apply(get_lowest_page_number)
    forms_df = forms_df.dropna(subset=['lowest_page'])
    forms_df['lowest_page'] = forms_df['lowest_page'].astype(int)
    logger.info(f"Documents with valid page numbers: {len(forms_df)}")

# Check processing status
logger.info(f"\n{'='*50}")
logger.info("CHECKPOINT STATUS")
logger.info(f"{'='*50}")

if TESTING_MODE:
    logger.info("🧪 MODE: TESTING")
    logger.info(f"Source: {TESTING_DIR}")
else:
    logger.info("🚀 MODE: PRODUCTION")
    if FILTER_AGENCIES is not None:
        if isinstance(FILTER_AGENCIES, str):
            logger.info(f"🔍 Filter: {FILTER_AGENCIES}")
        else:
            logger.info(f"🔍 Filter: {', '.join(FILTER_AGENCIES)}")

logger.info(f"Total documents queued: {len(forms_df)}")
logger.info(f"DPI: {DPI}")

# Create output directory and check existing
output_dir = Path(ACTIVE_OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

already_processed = 0
for idx, row in forms_df.iterrows():
    filename_base = Path(row['filename']).stem
    json_filename = f"{filename_base}_page_{row['lowest_page']}_donut_improved.json"
    json_path = output_dir / json_filename
    if json_path.exists() and json_path.stat().st_size > 0:
        already_processed += 1

remaining = len(forms_df) - already_processed
logger.info(f"Already completed: {already_processed} files")
logger.info(f"Remaining to process: {remaining} files")
logger.info(f"Progress: {(already_processed/len(forms_df)*100):.1f}% complete")

if not CLOBBER and already_processed > 0:
    logger.info("💡 TIP: Will skip already processed files (CLOBBER=False)")
    logger.info("💡 To reprocess: Set CLOBBER=True")

logger.info(f"{'='*50}\n")

## Process Documents with Progress Bar

In [ ]:
# Initialize counters
processed_count = 0
skipped_count = 0
error_count = 0
validation_failures = 0

logger.info("Starting document processing...\n")

# Create progress bar
progress_bar = tqdm(
    forms_df.iterrows(),
    total=len(forms_df),
    desc="Processing forms",
    unit="file",
    disable=False  # Set to True to completely disable progress bar
)

for idx, row in progress_bar:
    filename = row['filename']
    lowest_page = row['lowest_page']
    
    # Update progress bar description with current file
    progress_bar.set_postfix_str(f"{filename[:30]}...")
    
    # Generate output filename
    filename_base = Path(filename).stem
    json_filename = f"{filename_base}_page_{lowest_page}_donut_improved.json"
    json_path = output_dir / json_filename
    
    # Check if already processed
    if not CLOBBER and json_path.exists() and json_path.stat().st_size > 0:
        skipped_count += 1
        continue
    
    # Construct PDF path
    if TESTING_MODE:
        pdf_path = Path(row['full_path'])
    else:
        pdf_path = Path(CONTRACTS_DIR) / filename
    
    if not pdf_path.exists():
        logger.error(f"PDF not found: {filename}")
        error_count += 1
        continue
    
    try:
        # Convert PDF page to image
        image = pdf_page_to_image(str(pdf_path), lowest_page, dpi=DPI)
        
        # Process with hierarchical extraction
        results = process_with_donut_hierarchical(image, filename)
        
        # Add metadata
        results['source_file'] = filename
        results['source_page'] = lowest_page
        
        # Save results
        with open(json_path, 'w') as f:
            json.dump(results, f, indent=2)
        
        # Validate saved JSON
        if not validate_json_output(json_path):
            validation_failures += 1
        
        processed_count += 1
        
        # Log summary if verbose
        if VERBOSE and 'structured_data' in results:
            sd = results['structured_data']
            logger.info(f"  EDS: {sd.get('eds_number', 'N/A')} | "
                       f"Agency: {sd.get('agency', {}).get('name', 'N/A')} | "
                       f"Vendor: {sd.get('vendor', {}).get('name', 'N/A')}")
        
        # Clear GPU cache to prevent memory leaks
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
        
    except Exception as e:
        logger.error(f"Error processing {filename}: {str(e)}")
        error_count += 1
        
        # Save error info
        error_data = {
            'source_file': filename,
            'source_page': lowest_page,
            'error': str(e),
            'processing_timestamp': datetime.now().isoformat(),
            'testing_mode': TESTING_MODE
        }
        with open(json_path, 'w') as f:
            json.dump(error_data, f, indent=2)

# Close progress bar
progress_bar.close()

# Final summary
logger.info(f"\n{'='*50}")
logger.info("PROCESSING COMPLETE")
logger.info(f"{'='*50}")
logger.info(f"✓ Processed: {processed_count}")
logger.info(f"⊘ Skipped (already done): {skipped_count}")
logger.info(f"✗ Errors: {error_count}")
if validation_failures > 0:
    logger.warning(f"⚠ Validation failures: {validation_failures}")
logger.info(f"Total: {processed_count + skipped_count + error_count}")

if TESTING_MODE:
    logger.info(f"\n📁 Results: {TESTING_OUTPUT_DIR}")
else:
    logger.info(f"\n📁 Results: {OUTPUT_DIR}")

logger.info(f"📝 Log file: {log_filename}")
logger.info(f"{'='*50}")

## Display Sample Results

In [ ]:
# Load and display a sample result
json_files = list(output_dir.glob("*_donut_improved.json"))

if json_files:
    sample_file = json_files[0]
    logger.info(f"\nSample result: {sample_file.name}")
    
    with open(sample_file, 'r') as f:
        sample_data = json.load(f)
    
    # Display metadata
    print("\n=== METADATA ===")
    print(f"Source: {sample_data.get('source_file', 'N/A')}")
    print(f"Page: {sample_data.get('source_page', 'N/A')}")
    print(f"DPI: {sample_data.get('dpi', 'N/A')}")
    print(f"Testing Mode: {sample_data.get('testing_mode', 'N/A')}")
    print(f"Timestamp: {sample_data.get('processing_timestamp', 'N/A')}")
    
    # Display key extracted data
    if 'structured_data' in sample_data:
        print("\n=== KEY EXTRACTED DATA ===")
        sd = sample_data['structured_data']
        print(f"EDS Number: {sd.get('eds_number', 'N/A')}")
        print(f"Date Prepared: {sd.get('date_prepared', 'N/A')}")
        print(f"Agency: {sd.get('agency', {}).get('name', 'N/A')}")
        print(f"Vendor: {sd.get('vendor', {}).get('name', 'N/A')}")
        
        amount = sd.get('fiscal', {}).get('amount_this_action')
        if amount:
            print(f"Amount: ${amount:,.2f}")
        
        from_date = sd.get('time_period', {}).get('from_date')
        to_date = sd.get('time_period', {}).get('to_date')
        if from_date and to_date:
            print(f"Period: {from_date} to {to_date}")
        
        print("\nFor full structured data, open the JSON file")
else:
    logger.warning("No results found yet.")

## Create Summary Statistics

In [ ]:
# Analyze all processed files
json_files = list(output_dir.glob("*_donut_improved.json"))

if json_files:
    logger.info(f"\n{'='*50}")
    logger.info("SUMMARY STATISTICS")
    logger.info(f"{'='*50}")
    logger.info(f"Mode: {'TESTING' if TESTING_MODE else 'PRODUCTION'}")
    logger.info(f"Total files: {len(json_files)}")
    
    # Collect statistics
    stats = {
        'total_files': len(json_files),
        'files_with_errors': 0,
        'contract_types': {},
        'source_selection_methods': {},
        'minority_owned': 0,
        'women_owned': 0,
        'total_amount': 0.0,
        'files_with_amount': 0
    }
    
    for json_file in json_files:
        with open(json_file, 'r') as f:
            data = json.load(f)
        
        if 'error' in data:
            stats['files_with_errors'] += 1
            continue
        
        if 'structured_data' not in data:
            continue
        
        sd = data['structured_data']
        
        # Contract types
        if 'contract_info' in sd:
            for key, value in sd['contract_info'].items():
                if value is True:
                    stats['contract_types'][key] = stats['contract_types'].get(key, 0) + 1
        
        # Source selection
        if 'source_selection' in sd:
            for key, value in sd['source_selection'].items():
                if value is True and not key.endswith('_details'):
                    stats['source_selection_methods'][key] = stats['source_selection_methods'].get(key, 0) + 1
        
        # Vendor diversity
        if sd.get('vendor', {}).get('minority_owned'):
            stats['minority_owned'] += 1
        if sd.get('vendor', {}).get('women_owned'):
            stats['women_owned'] += 1
        
        # Amounts
        amount = sd.get('fiscal', {}).get('amount_this_action')
        if amount:
            stats['total_amount'] += amount
            stats['files_with_amount'] += 1
    
    logger.info(f"\nFiles with errors: {stats['files_with_errors']}")
    
    if stats['contract_types']:
        logger.info(f"\nContract Types (top 5):")
        for ctype, count in sorted(stats['contract_types'].items(), key=lambda x: x[1], reverse=True)[:5]:
            logger.info(f"  {ctype}: {count}")
    
    if stats['source_selection_methods']:
        logger.info(f"\nSource Selection Methods:")
        for method, count in sorted(stats['source_selection_methods'].items(), key=lambda x: x[1], reverse=True):
            logger.info(f"  {method}: {count}")
    
    logger.info(f"\nVendor Diversity:")
    logger.info(f"  Minority-owned: {stats['minority_owned']}")
    logger.info(f"  Women-owned: {stats['women_owned']}")
    
    logger.info(f"\nFiscal Information:")
    logger.info(f"  Files with amount data: {stats['files_with_amount']}")
    if stats['files_with_amount'] > 0:
        logger.info(f"  Total contract amount: ${stats['total_amount']:,.2f}")
        logger.info(f"  Average contract amount: ${stats['total_amount']/stats['files_with_amount']:,.2f}")
    
    logger.info(f"{'='*50}")
else:
    logger.warning("No processed files found.")

## Configuration Reference

### Key Settings:

```python
# For quick testing on 3-5 files:
TESTING_MODE = True
TESTING_SAMPLE_SIZE = 5
VERBOSE = True  # See detailed logs

# For production runs:
TESTING_MODE = False
VERBOSE = False  # Quiet operation
FILTER_AGENCIES = None  # Or specify agencies

# To reprocess existing files:
CLOBBER = True
```

### Features:

✅ **Progress Bar** - Visual progress with ETA  
✅ **Logging** - All output saved to timestamped log file  
✅ **Resume** - Automatically skip processed files  
✅ **Memory Management** - GPU cache clearing after each file  
✅ **Validation** - JSON integrity checks  
✅ **Quiet Mode** - Minimal console output (VERBOSE=False)  

### Logs Location:
- All logs saved to: `../../logs/donut_processing_YYYYMMDD_HHMMSS.log`
- Review later for debugging or auditing